# 🟦 Familia 1 · 1D Convolutional Neural Networks
**Detección de Parkinson en voz — TFM**

---
Este notebook entrena y evalúa los tres modelos de la familia CNN 1D:
- **`baseline_cnn1d`** — CNN estándar de referencia
- **`residual_cnn1d`** — CNN profunda con residual connections
- **`dilated_cnn1d`** — CNN con convoluciones dilatadas

**Flujo de trabajo:**
1. Configura el modelo y los hiperparámetros en la celda `⚙️ CONFIGURACION`
2. Ejecuta todas las celdas (`Kernel → Restart & Run All`)
3. Los resultados quedan guardados automáticamente en `experiments/`

## ⚙️ CONFIGURACION — Edita aquí antes de ejecutar

In [ ]:
# ═══════════════════════════════════════════════════
#  PARAMETROS PRINCIPALES  ← EDITAR AQUÍ
# ═══════════════════════════════════════════════════

MODEL_NAME   = "baseline_cnn1d"   # Opciones: "baseline_cnn1d" | "residual_cnn1d" | "dilated_cnn1d"
DATASET_NAME = "neurovoz"          # Opciones: "neurovoz" | "pc-gita"
FOLD_INDEX   = 0                   # Fold a usar: 0, 1, 2, 3 ó 4

# ─── Hiperparámetros del modelo ────────────────────
# Cambia estos valores para probar variantes sin tocar el código
MODEL_HYPERPARAMS = {
    # === Para baseline_cnn1d ===
    "n_filters":   [32, 64, 128],  # Número de filtros por bloque
    "kernel_size": 7,              # Tamaño del kernel
    "dropout":     0.3,            # Tasa de dropout

    # === Para residual_cnn1d (ignorados por baseline) ===
    "n_res_blocks": 2,             # Residual blocks por stage

    # === Para dilated_cnn1d (ignorados por los otros) ===
    "n_channels": 64,              # Canales uniformes en todos los bloques
    "dilations":  [1, 2, 4, 8, 16], # Factores de dilatación
}

# ─── Hiperparámetros de entrenamiento ──────────────
TRAIN_CONFIG = {
    "epochs":      50,
    "batch_size":  32,
    "lr":          1e-3,
    "weight_decay": 1e-4,
    "patience":    10,       # Early stopping: nº de epochs sin mejora
    "augment":     True,     # Data augmentation en train
}

# ─── Nota libre (se guarda en el log) ──────────────
NOTES = "Prueba inicial baseline_cnn1d fold 0"

print(f"Modelo: {MODEL_NAME} | Dataset: {DATASET_NAME} | Fold: {FOLD_INDEX + 1}")
print(f"Config: {TRAIN_CONFIG}")

## 1. Imports y configuración de entorno

In [ ]:
import sys, os
# Asegura que Python encuentra los modulos del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from src.models.cnn_1d import CNN_MODELS
from src.training.audio_dataset import get_dataloaders, get_class_weights
from src.training.trainer import Trainer
from src.training.experiment_logger import ExperimentLogger
from src.evaluation.metrics import calculate_metrics, plot_roc_curve

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Carga de datos

In [ ]:
SPLITS_JSON = os.path.join(PROJECT_ROOT, "data", "data_splits.json")
DATA_ROOT   = os.path.join(PROJECT_ROOT, "data", "processed")

train_loader, val_loader, test_loader = get_dataloaders(
    splits_json  = SPLITS_JSON,
    dataset_name = DATASET_NAME,
    fold_index   = FOLD_INDEX,
    data_root    = DATA_ROOT,
    batch_size   = TRAIN_CONFIG["batch_size"],
    augment_train= TRAIN_CONFIG["augment"],
)

# Pesos para desbalanceo de clases
import json
with open(SPLITS_JSON) as f:
    splits = json.load(f)
train_files = splits[DATASET_NAME][FOLD_INDEX]["train_files"]
class_weights = get_class_weights(train_files).to(DEVICE)
print(f"Pesos de clase: Control={class_weights[0]:.3f} | Parkinson={class_weights[1]:.3f}")

# Verificar un batch
batch_x, batch_y = next(iter(train_loader))
print(f"\nBatch shape: {batch_x.shape} | Labels: {batch_y[:8].tolist()}")

## 3. Construcción del modelo

In [ ]:
# Construye el modelo con los hiperparámetros definidos en la celda de configuración
ModelClass = CNN_MODELS[MODEL_NAME]

if MODEL_NAME == "baseline_cnn1d":
    model = ModelClass(
        n_filters   = MODEL_HYPERPARAMS["n_filters"],
        kernel_size = MODEL_HYPERPARAMS["kernel_size"],
        dropout     = MODEL_HYPERPARAMS["dropout"],
        num_classes = 2,
    )
elif MODEL_NAME == "residual_cnn1d":
    model = ModelClass(
        n_filters     = MODEL_HYPERPARAMS["n_filters"],
        kernel_size   = MODEL_HYPERPARAMS["kernel_size"],
        n_res_blocks  = MODEL_HYPERPARAMS["n_res_blocks"],
        dropout       = MODEL_HYPERPARAMS["dropout"],
        num_classes   = 2,
    )
elif MODEL_NAME == "dilated_cnn1d":
    model = ModelClass(
        n_channels  = MODEL_HYPERPARAMS["n_channels"],
        kernel_size = MODEL_HYPERPARAMS["kernel_size"],
        dilations   = MODEL_HYPERPARAMS["dilations"],
        dropout     = MODEL_HYPERPARAMS["dropout"],
        num_classes = 2,
    )

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo: {MODEL_NAME}")
print(f"Parámetros entrenables: {n_params:,}")
print(model)

## 4. Entrenamiento con Early Stopping

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=TRAIN_CONFIG["lr"],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=5, factor=0.5, verbose=True
)

trainer = Trainer(model, optimizer, criterion, DEVICE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

print(f"Iniciando entrenamiento: {TRAIN_CONFIG['epochs']} epochs máximo")
print("-" * 60)

for epoch in range(TRAIN_CONFIG["epochs"]):
    train_loss, train_preds, train_labels = trainer.train_epoch(train_loader)
    val_loss, val_preds, val_probs, val_labels = trainer.evaluate(val_loader)

    train_acc = sum(p == l for p, l in zip(train_preds, train_labels)) / len(train_labels)
    val_acc   = sum(p == l for p, l in zip(val_preds, val_labels))   / len(val_labels)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:03d}/{TRAIN_CONFIG['epochs']} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} "
          f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= TRAIN_CONFIG["patience"]:
            print(f"\nEarly stopping en epoch {epoch+1}")
            break

# Restaurar mejor modelo
model.load_state_dict(best_model_state)
print(f"\nMejor Val Loss: {best_val_loss:.4f} — Modelo restaurado.")

## 5. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train", color="steelblue")
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   color="tomato")
axes[0].set_title("Loss por Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train", color="steelblue")
axes[1].plot(epochs_range, history["val_acc"],   label="Val",   color="tomato")
axes[1].set_title("Accuracy por Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f"{MODEL_NAME.upper()} — {DATASET_NAME} Fold {FOLD_INDEX+1}", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Evaluación en Test Set

In [ ]:
_, test_preds, test_probs, test_labels = trainer.evaluate(test_loader)
metrics = calculate_metrics(test_labels, test_preds, test_probs)

print("\n" + "="*50)
print(f" METRICAS EN TEST SET ({DATASET_NAME} — Fold {FOLD_INDEX+1})")
print("="*50)
for k, v in metrics.items():
    print(f"  {k:<20}: {v:.4f}")

## 7. Guardar modelo y registrar experimento

In [ ]:
# ── Guardar pesos del modelo ──────────────────────────
MODELS_SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "saved_models")
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)

import datetime
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"{ts}_{MODEL_NAME}_{DATASET_NAME}_fold{FOLD_INDEX+1}.pt"
model_save_path = os.path.join(MODELS_SAVE_DIR, model_filename)

torch.save({
    "model_name":  MODEL_NAME,
    "hyperparams": MODEL_HYPERPARAMS,
    "state_dict":  model.state_dict(),
    "metrics":     metrics,
    "dataset":     DATASET_NAME,
    "fold":        FOLD_INDEX + 1,
}, model_save_path)
print(f"Modelo guardado en: {model_save_path}")

# ── Registrar en el log de experimentos ───────────────
LOG_DIR = os.path.join(PROJECT_ROOT, "experiments")
logger = ExperimentLogger(log_dir=LOG_DIR)

run_id = logger.log(
    model_name  = MODEL_NAME,
    dataset     = DATASET_NAME,
    fold        = FOLD_INDEX + 1,
    hyperparams = {**MODEL_HYPERPARAMS, **TRAIN_CONFIG},
    metrics     = metrics,
    history     = history,
    notes       = NOTES,
)

# Mostrar resumen de todos los experimentos hasta ahora
logger.print_summary()